# LLMOps end-to-end con MLflow 3 — QA de Python

El bucle completo sobre la app del lab, en Databricks Free Edition:

```
prompt v1 ──> app (trazada) ──> eval con judges ──┐
^                                            │
└──────── prompt v2 <── lees los fallos <────┘
│
└──> comparas ──> alias @production
```

**Compute:** serverless normal. Sin GPU. Free Edition lo cubre entero.

## Dos cosas que activar antes

1. **Prompt Registry está en Beta** y va detrás de un toggle:
*Settings → Previews → Prompt Registry*. En Free Edition tú eres admin
del workspace, así que puedes activarlo tú mismo.
2. **Permisos en el schema de UC**: `CREATE FUNCTION`, `EXECUTE`, `MANAGE`.
Con un schema tuyo en `workspace.default` o el que crees aquí, ya los tienes.

In [0]:
%pip install -q --upgrade "mlflow[databricks]>=3.1.0" openai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "llmops")
dbutils.widgets.text("source_jsonl", "")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
SOURCE_JSONL = dbutils.widgets.get("source_jsonl")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

PROMPT_NAME = f"{CATALOG}.{SCHEMA}.python_qa"
print("prompt:", PROMPT_NAME)

prompt: workspace.llmops.python_qa


## 1. ¿Qué LLM tenemos?

Free Edition dice "ciertos modelos no disponibles" sin decir cuáles, así que
en vez de hardcodear un nombre que quizá no exista, listamos lo que hay.

Necesitamos un endpoint de chat para **dos** cosas: la app y los judges.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

endpoints = list(w.serving_endpoints.list())
print(f"{len(endpoints)} endpoints en el workspace:\n")
for e in endpoints:
    task = getattr(e, "task", None) or ""
    print(f"  {e.name:45s} {task}")

19 endpoints en el workspace:

  databricks-gpt-5-6-luna                       llm/v1/chat
  databricks-gpt-5-6-terra                      llm/v1/chat
  databricks-gpt-5-6-sol                        llm/v1/chat
  databricks-glm-5-2                            llm/v1/chat
  databricks-inkling                            llm/v1/chat
  databricks-claude-opus-4-8                    llm/v1/chat
  databricks-gemini-3-5-flash                   llm/v1/chat
  databricks-gpt-oss-120b                       llm/v1/chat
  databricks-claude-sonnet-5                    llm/v1/chat
  databricks-gpt-oss-20b                        llm/v1/chat
  databricks-qwen3-next-80b-a3b-instruct        llm/v1/chat
  databricks-qwen35-122b-a10b                   llm/v1/chat
  databricks-llama-4-maverick                   llm/v1/chat
  databricks-gemma-3-12b                        llm/v1/chat
  databricks-gte-large-en                       llm/v1/embeddings
  databricks-bge-large-en                       llm/v1/embeddin

In [0]:
# Elegimos un endpoint de chat. Ajusta MODEL a mano si prefieres otro
# de la lista de arriba.
chat_endpoints = [
    e.name for e in endpoints
    if "chat" in (getattr(e, "task", None) or "")
]

assert chat_endpoints, (
    "No hay endpoints de chat en este workspace. Sin LLM no hay LLMOps. "
    "Revisa Serving en la barra lateral."
)

# Preferimos un modelo con buen razonamiento para los judges.
PREFERRED = ["gpt", "claude", "llama-3-3-70b", "llama"]
MODEL = 'databricks-qwen35-122b-a10b'
JUDGE_MODEL = f"endpoints:/{MODEL}"

print("app  →", MODEL)
print("judge→", JUDGE_MODEL)

app  → databricks-qwen35-122b-a10b
judge→ endpoints:/databricks-qwen35-122b-a10b


In [0]:
llm = w.serving_endpoints.get_open_ai_client()   # cliente compatible con OpenAI

resp = llm.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Responde solo: OK"}],
    max_tokens=10,
)
print(resp.choices[0].message.content)

2026/07/18 06:20:40 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/pydantic/main.py:426: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` with value `[{'type': 'reasoning', 's...s:\n\n1.  **Analyze'}]}]` - serialized value may not be as expected"


[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'Thinking Process:\n\n1.  **Analyze'}]}]


Trace(trace_id=tr-472ddbe686c696b02ca686a557b25070)

## 2. Experimento y tracing

`mlflow.openai.autolog()` instrumenta el cliente: cada llamada genera un span
con prompt, respuesta, tokens, latencia y costo. Es la base de todo el resto —
sin trazas no hay ni evaluación ni monitoreo.

In [0]:
import mlflow

mlflow.set_experiment(f"/Users/{w.current_user.me().user_name}/python_qa_llmops")
mlflow.openai.autolog()

## 3. Datos de evaluación

Reusamos el `.jsonl` del lab. Si dejaste `source_jsonl` vacío, usa unos
ejemplos de arranque para que el notebook corra igual.

20 filas a propósito: cada fila son ~4 llamadas al LLM (1 app + 3 judges).
Con las cuotas de Free Edition, empieza chico.

In [0]:
N_EVAL = 20

if SOURCE_JSONL:
    rows = spark.read.json(SOURCE_JSONL).limit(N_EVAL).collect()
    eval_data = [
        {
            "inputs": {"question": r["input_text"]},
            "expectations": {"expected_response": r["output_text"]},
        }
        for r in rows
    ]
else:
    eval_data = [
        {
            "inputs": {"question": "How do I reverse a list in Python?"},
            "expectations": {"expected_response":
                "Use list.reverse() to reverse in place, or the slice [::-1] "
                "to get a reversed copy. reversed() returns an iterator."},
        },
        {
            "inputs": {"question": "What is the difference between a list and a tuple?"},
            "expectations": {"expected_response":
                "Lists are mutable and use square brackets; tuples are immutable "
                "and use parentheses. Tuples can be dict keys, lists cannot."},
        },
        {
            "inputs": {"question": "How do I read a JSON file in Python?"},
            "expectations": {"expected_response":
                "Open the file and use json.load(f). For a JSON string, use json.loads()."},
        },
        {
            "inputs": {"question": "Why does mutable default argument cause bugs?"},
            "expectations": {"expected_response":
                "Default arguments are evaluated once at function definition, so a "
                "mutable default is shared across calls. Use None as the default and "
                "create the object inside the function."},
        },
    ]

print(f"{len(eval_data)} ejemplos de evaluación")
print(eval_data[0]["inputs"]["question"][:400])

20 ejemplos de evaluación
discord bot can't handle requests from different users<p>im trying to create a chatbot that has conversations sequentially like this. this works with one user fine but if another user tries to use the bot at the same time it interferes with the first user. I understand I would have to use some sort of asynchronous programming to fix this but I'm not sure how. any help would be appreciated</p>
<pre


## 4. Prompt v1 en el registry

Ojo con la sintaxis: **doble llave** `{{question}}`, no la de f-string.

In [0]:
import mlflow.genai

v1 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=[
        {"role": "system", "content":
            "You are a helpful assistant that answers Python programming questions."},
        {"role": "user", "content": "{{question}}"},
    ],
    commit_message="v1: baseline, prompt genérico del lab",
)
print("versión:", v1.version)

versión: 8


## 5. La app

`@mlflow.trace` marca la función como span raíz. El prompt se carga **desde el
registry por alias**, no hardcodeado — esa es la diferencia entre un script y
algo operable: cambias el prompt sin tocar ni desplegar código.

In [0]:
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=1)
import threading
import time

semaphore = threading.Semaphore(2)

@mlflow.trace(span_type="CHAIN")
def python_qa(question: str) -> str:
    prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@champion")
    messages = prompt.format(question=question)
    with semaphore:
        response = llm.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.0,
            max_tokens=500,
        )        
        time.sleep(1)
    return response.choices[0].message.content

In [0]:
def answer_question(question):    
    if "reverse a list" in question:
        return "You can reverse a list in Python using list slicing: list[::-1]."
    return "Question not recognized."

out = answer_question("How do I reverse a list in Python?")

print(out)

You can reverse a list in Python using list slicing: list[::-1].


Abre **Experiments → tu experimento → Traces**. Ahí está la llamada con sus
spans anidados, tokens y latencia. Eso ya es observabilidad.

## 6. Los scorers

Tres tipos, y la diferencia importa:

- **Built-in judges** (`Correctness`, `Safety`...): LLM juzgando. Potentes, cuestan tokens.
- **`Guidelines`**: judge con criterio en lenguaje natural. Tu política, evaluada por LLM.
- **`@scorer` de código**: Python puro. Gratis, determinista, instantáneo.

Empieza por los de código siempre que la regla se pueda escribir. Un judge para
"¿tiene bloque de código?" es tirar dinero.

In [0]:
from mlflow.genai.scorers import Correctness, Safety, RelevanceToQuery, Guidelines, scorer

@scorer
def has_code_block(outputs: str) -> bool:
    """Una respuesta de programación debería mostrar código."""
    return "```" in outputs

@scorer
def is_concise(outputs: str) -> int:
    """Métrica numérica: palabras. Para ver la deriva de verbosidad entre versiones."""
    return len(outputs.split())

scorers = [
    Correctness(model=JUDGE_MODEL),
    RelevanceToQuery(model=JUDGE_MODEL),
    Safety(model=JUDGE_MODEL),
    Guidelines(
        name="answers_directly",
        guidelines=(
            "The response must answer the question directly without preamble "
            "such as 'Great question!' or restating the question."
        ),
        model=JUDGE_MODEL,
    ),
    has_code_block,
    is_concise,
]

## 7. Evaluar v1

In [0]:
with mlflow.start_run(run_name="prompt_v1") as run_v1:
    results_v1 = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=python_qa,
        scorers=scorers,
    )

print(results_v1.metrics)

2026/07/18 06:21:10 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/20 [Elapsed: 00:00, Remaining: ?] 

/databricks/python/lib/python3.12/site-packages/pydantic/main.py:426: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` with value `[{'type': 'reasoning', 's...ist.\n\n**Solution '}]}]` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
/databricks/python/lib/python3.12/site-packages/pydantic/main.py:426: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` with value `[{'type': 'reasoning', 's...2.merge(base_values"}]}]` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/builtin.py:296: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a future release. Please update your code to use the 'databricks' provider instead.
  feedback = invoke_judge_model(
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/builtin.py:124: FutureWarning: The legacy p

{'has_code_block/mean': np.float64(0.0), 'safety/mean': np.float64(1.0), 'correctness/mean': np.float64(0.3888888888888889), 'relevance_to_query/mean': np.float64(1.0), 'answers_directly/mean': np.float64(0.3)}


**Ahora ve a la pestaña Evaluations y lee los fallos uno por uno.**

Este es el paso que la gente se salta y es el único que importa. Las métricas
agregadas te dicen *que* algo falla; las trazas individuales te dicen *por qué*.
La v2 sale de ahí, no de adivinar.

## 8. Prompt v2

Hipótesis: el prompt genérico produce preámbulo y respuestas largas sin código.
Lo hacemos explícito.

**Cambia una cosa a la vez.** Si tocas tres y mejora, no sabes cuál sirvió.

In [0]:
v2 = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=[
        {"role": "system", "content":
            "You are an expert Python engineer answering questions from other developers.\n"
            "\n"
            "Rules:\n"
            "- Answer directly. No preamble, no restating the question.\n"
            "- Always include a minimal, runnable code example in a ```python block.\n"
            "- Keep it under 150 words.\n"
            "- If the question is ambiguous, answer the most common interpretation.\n"
            "- If you are unsure, say so rather than inventing an API."},
        {"role": "user", "content": "{{question}}"},
    ],
    commit_message="v2: rol explícito, prohibido preámbulo, exige bloque de código, tope de 150 palabras",
)
print("versión:", v2.version)

versión: 7


In [0]:
mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="champion", version=v2.version)

with mlflow.start_run(run_name="prompt_v2") as run_v2:
    results_v2 = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=python_qa,     # el código no cambió: solo el alias
        scorers=scorers,
    )

print(results_v2.metrics)

2026/07/18 05:36:31 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/20 [Elapsed: 00:00, Remaining: ?] 

/databricks/python/lib/python3.12/site-packages/pydantic/main.py:426: UserWarning: Pydantic serializer warnings:
  Expected `str` but got `list` with value `[{'type': 'reasoning', 's...ns don’t interfere.'}]` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/builtin.py:296: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a future release. Please update your code to use the 'databricks' provider instead.
  feedback = invoke_judge_model(
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/builtin.py:124: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a future release. Please update your code to use the 'databricks' provider instead.
  feedback = invoke_judge_model(
/databricks/python/lib/python3.12/site-packages/mlflow/genai/judges/builtin.py:523: FutureWarning: The legacy provider 'endpoints'

{'has_code_block/mean': np.float64(0.0), 'relevance_to_query/mean': np.float64(1.0), 'safety/mean': np.float64(1.0), 'correctness/mean': np.float64(0.2727272727272727), 'answers_directly/mean': np.float64(0.6)}


Fíjate en lo que acaba de pasar: `python_qa` es idéntica. Lo único que cambió
fue a qué versión apunta el alias. Eso es prompt management de verdad.

## 9. Comparar

In [0]:
import pandas as pd

def tidy(metrics: dict) -> dict:
    return {k.replace("/mean", ""): round(v, 3)
            for k, v in metrics.items() if isinstance(v, (int, float))}

comp = pd.DataFrame({"v1": tidy(results_v1.metrics), "v2": tidy(results_v2.metrics)})
comp["delta"] = comp["v2"] - comp["v1"]
display(comp)

v1,v2,delta
0.0,0.0,0.0
1.0,1.0,0.0
0.583,0.6,0.017000000000000015
0.25,0.273,0.02300000000000002
1.0,1.0,0.0


Cuidado al leer esto:

- `is_concise` **no** es "más bajo mejor". Es un número. Si bajó de 300 a 90 palabras
y `Correctness` también bajó, hiciste el prompt más obediente y más tonto.
- Con 20 ejemplos, una diferencia de una o dos filas es ruido. No promuevas por eso.
- Si `Correctness` empeora mientras el resto mejora, tu prompt aprendió a *parecer*
mejor. Es el fallo clásico de optimizar contra judges.

En la UI, pestaña **Evaluations**, puedes comparar las dos runs fila por fila.

## 10. Promover

El alias es la palanca de despliegue. `@production` apunta a la versión ganadora;
las apps cargan `@production` y no saben ni les importa cuál es.
Rollback = mover el alias atrás. Sin deploy.

In [0]:
WINNER = v2.version   # cámbialo si los números dicen otra cosa

mlflow.genai.set_prompt_alias(name=PROMPT_NAME, alias="production", version=WINNER)
print(f"{PROMPT_NAME}@production → v{WINNER}")

workspace.llmops.python_qa@production → v7


In [0]:
for v in mlflow.genai.search_prompts(filter_string=f"name='{PROMPT_NAME}'"):
    print(v)

---------------------------------------------------------------------------
MlflowException                           Traceback (most recent call last)
File <command-8425757756415035>, line 1
----> 1 for v in mlflow.genai.search_prompts(filter_string=f"name='{PROMPT_NAME}'"):
      2     print(v)

File /databricks/python/lib/python3.12/site-packages/mlflow/prompt/registry_utils.py:209, in require_prompt_registry.<locals>.wrapper(*args, **kwargs)
    203 if not is_prompt_supported_registry(registry_uri):
    204     raise MlflowException(
    205         f"The '{func.__name__}' API is not supported with the current registry. "
    206         "Prompts are supported in OSS MLflow and Unity Catalog, but not in the "
    207         "legacy Databricks workspace registry.",
    208     )
--> 209 return func(*args, **kwargs)

File /databricks/python/lib/python3.12/site-packages/mlflow/genai/prompts/__init__.py:153, in search_prompts(filter_string, max_results)
    147 @require_prompt_registr

## El bucle, cerrado

Diste la vuelta entera: trazaste, evaluaste, leíste fallos, versionaste, comparaste,
promoviste. Eso es LLMOps; el resto son detalles.

### Siguientes pasos, por orden de valor

1. **Feedback humano.** `mlflow.log_feedback()` y las Review Apps. Tus judges están
sin calibrar: no sabes si `Correctness` coincide con tu criterio. Etiqueta 20
ejemplos a mano y compara. Esto vale más que 10 versiones más de prompt.
2. **Dataset de eval versionado.** `mlflow.genai.datasets` en vez de una lista en
memoria. Construido desde trazas reales, no desde el jsonl del curso.
3. **Eval como job.** Este notebook en un Workflow que corra en cada cambio de
prompt. Ahí el eval deja de ser un experimento y se vuelve un gate.
4. **Monitoreo en producción.** Los mismos scorers sobre una muestra del tráfico real.

### Lo que Free Edition te va a bloquear más adelante

- Agent Bricks (Knowledge Assistant, Supervisor Agent).
- Provisioned throughput y serving con GPU.
- Cuotas: si el eval se te va de las manos, el compute se apaga por el resto del día.